In [1]:
"""
DeePiCt - ribosome 2-fold training (mirrors deepict hsp60_depth2 workflow +
deepETpicker ribosome split definitions).
"""
from pathlib import Path
import json, os, shutil, subprocess, time

import pandas as pd
import yaml

repo = Path("./release_assets/external/DeePiCt/3d_cnn")
metadata_csv = Path("./release_assets/external/DeePiCt/3d_cnn/metadata_dxd20240920_ribo.csv")
base_out = Path("/virtual_data/train_data/training/deepict/two_fold_ribosome_depth2")
configs_dir = base_out / "configs"
results_dir = base_out / "results"
configs_dir.mkdir(parents=True, exist_ok=True)
results_dir.mkdir(parents=True, exist_ok=True)
pythonpath = repo / "src"

# Same split as deepETpicker/ribosome.ipynb
fold1 = [
    "24092009",
    "24092014",
    "24092017",
    "24092023",
    "24092025",
    "24092029",
    "24092031",
]
fold2 = [
    "24092002",
    "24092013",
    "24092016",
    "24092020",
    "24092024",
    "24092026",
    "24092030",
]

fold_specs = [
    {
        "name": "fold1_train_fold2_test",
        "training_list": fold1,
        "prediction_list": fold2,
        "gpu_id": "0",
    },
    {
        "name": "fold2_train_fold1_test",
        "training_list": fold2,
        "prediction_list": fold1,
        "gpu_id": "0",
    },
]

# Sanity-check metadata coverage and required columns before starting long runs
metadata_df = pd.read_csv(metadata_csv)
required_cols = {"tomo_name", "tomo", "ribosome_mask"}
missing_cols = required_cols - set(metadata_df.columns)
if missing_cols:
    raise ValueError(f"Missing required metadata columns: {sorted(missing_cols)}")

all_fold_ids = sorted(fold1 + fold2)
if set(fold1).intersection(fold2):
    raise ValueError("fold1 and fold2 overlap")
if sorted(metadata_df["tomo_name"].astype(str).tolist()) != all_fold_ids:
    raise ValueError("metadata tomo_name values do not match the 2-fold split IDs")

# Ribosome-focused parameters (kept close to proven config_ribo defaults)
ribosome_params = {
    "box_size": 64,
    "overlap": 12,
    "batch_size": 5,
    "min_label_fraction": 0.001,
    "depth": 2,
    "initial_features": 4,
    "epochs": 18,
    "cluster_threshold": 0.6,
    "min_cluster_size": 200,
    "max_cluster_size": 10000,
}

# Start from ribosome template, then override only what we need for this run
base_config_path = repo / "config_ribo_fold1.yaml"
base_config = yaml.safe_load(base_config_path.read_text())
base_config["dataset_table"] = str(metadata_csv)
base_config["training"]["active"] = True
base_config["training"]["force_retrain"] = True
base_config["prediction"]["active"] = True

base_config["training"]["semantic_classes"] = ["ribosome"]
base_config["training"]["processing_tomo"] = "tomo"
base_config["training"]["box_size"] = ribosome_params["box_size"]
base_config["training"]["overlap"] = ribosome_params["overlap"]
base_config["training"]["batch_size"] = ribosome_params["batch_size"]
base_config["training"]["min_label_fraction"] = ribosome_params["min_label_fraction"]

unet = base_config["training"]["unet_hyperparameters"]
unet["depth"] = ribosome_params["depth"]
unet["initial_features"] = ribosome_params["initial_features"]
unet["epochs"] = ribosome_params["epochs"]

pp = base_config.setdefault("postprocessing_clustering", {})
if "min_cluster_size" not in pp:
    pp["min_cluster_size"] = pp.pop("min_clusterls_size", ribosome_params["min_cluster_size"])
pp["threshold"] = ribosome_params["cluster_threshold"]
pp["min_cluster_size"] = ribosome_params["min_cluster_size"]
pp["max_cluster_size"] = ribosome_params["max_cluster_size"]
pp["active"] = True

base_config["evaluation"]["segmentation_evaluation"]["active"] = True
base_config["evaluation"]["particle_picking"]["active"] = False

print("Using ribosome params:")
print(json.dumps(ribosome_params, indent=2))


def run_fold(spec):
    fold_name = spec["name"]
    training_list = spec["training_list"]
    prediction_list = spec["prediction_list"]

    output_dir = base_out / f"{fold_name}_out"
    work_dir = base_out / f"{fold_name}_work"
    model_path = base_out / f"{fold_name}_model.pth"
    config_path = configs_dir / f"{fold_name}.yaml"
    log_path = base_out / f"{fold_name}.log"

    cfg = json.loads(json.dumps(base_config))
    cfg["output_dir"] = str(output_dir)
    cfg["work_dir"] = str(work_dir)
    cfg["model_path"] = str(model_path)
    cfg["tomos_sets"]["training_list"] = training_list
    cfg["tomos_sets"]["prediction_list"] = prediction_list

    for d in [output_dir, work_dir]:
        if d.exists():
            shutil.rmtree(d)
    if model_path.exists():
        model_path.unlink()

    config_path.write_text(yaml.safe_dump(cfg, sort_keys=False))

    env = os.environ.copy()
    env["CUDA_VISIBLE_DEVICES"] = spec["gpu_id"]
    env["PYTHONPATH"] = str(pythonpath)

    start = time.time()
    with log_path.open("w") as f:
        proc = subprocess.run(
            [str(repo / "deploy_local.sh"), str(config_path)],
            cwd=repo,
            env=env,
            stdout=f,
            stderr=subprocess.STDOUT,
            check=False,
            text=True,
        )
    duration_sec = round(time.time() - start, 2)

    # Collect dice for predicted tomograms
    pp_stats_path = output_dir / "pp_statistics.csv"
    dice_by_tomo = {}
    if pp_stats_path.exists():
        pp_df = pd.read_csv(pp_stats_path)
        rows = pp_df[pp_df["statistic_variable"] == "dice"]
        for tomo_name in prediction_list:
            tomo_rows = rows[rows["tomo_name"] == tomo_name]
            if not tomo_rows.empty:
                dice_by_tomo[tomo_name] = float(tomo_rows.iloc[-1]["statistic_value"])

    copied_motl_files = []
    motl_files = list(output_dir.glob("predictions/**/*.csv")) if output_dir.exists() else []
    for mf in motl_files:
        if mf.name.startswith("motl_"):
            dst = results_dir / f"{fold_name}_{mf.name}"
            shutil.copy2(mf, dst)
            copied_motl_files.append(str(dst))

    return {
        "fold_name": fold_name,
        "training_list": training_list,
        "prediction_list": prediction_list,
        "config_path": str(config_path),
        "output_dir": str(output_dir),
        "model_path": str(model_path),
        "log_path": str(log_path),
        "pp_statistics": str(pp_stats_path),
        "prediction_coords": copied_motl_files,
        "gpu_id": spec["gpu_id"],
        "exit_code": proc.returncode,
        "duration_sec": duration_sec,
        "dice_by_tomo": dice_by_tomo,
        "mean_dice": (sum(dice_by_tomo.values()) / len(dice_by_tomo)) if dice_by_tomo else None,
    }


all_results = []
for spec in fold_specs:
    print("\n" + "=" * 80)
    print(f"Fold: {spec['name']}")
    print(f"Train: {spec['training_list']}")
    print(f"Predict/Test: {spec['prediction_list']}")
    result = run_fold(spec)
    all_results.append(result)
    print(f"  exit={result['exit_code']}, duration={result['duration_sec']}s, mean_dice={result['mean_dice']}")

results_json = base_out / "two_fold_results_depth2.json"
results_csv = base_out / "two_fold_results_depth2.csv"
results_json.write_text(json.dumps(all_results, indent=2))
pd.DataFrame(all_results).to_csv(results_csv, index=False)

results_df = pd.DataFrame(all_results)
print("\n" + "=" * 80)
print("FINAL RESULTS - ribosome 2-fold")
print(results_df[["fold_name", "exit_code", "duration_sec", "mean_dice"]])
failed = (results_df["exit_code"] != 0).sum()
print(f"\nTotal failed: {failed}/2")
if failed == 0:
    print("ALL 2 TESTS PASSED!")
print(f"\nSaved: {results_json}")
print(f"Saved: {results_csv}")

Using ribosome params:
{
  "box_size": 64,
  "overlap": 12,
  "batch_size": 5,
  "min_label_fraction": 0.001,
  "depth": 2,
  "initial_features": 4,
  "epochs": 18,
  "cluster_threshold": 0.6,
  "min_cluster_size": 200,
  "max_cluster_size": 10000
}

Fold: fold1_train_fold2_test
Train: ['24092009', '24092014', '24092017', '24092023', '24092025', '24092029', '24092031']
Predict/Test: ['24092002', '24092013', '24092016', '24092020', '24092024', '24092026', '24092030']
  exit=0, duration=3203.26s, mean_dice=None

Fold: fold2_train_fold1_test
Train: ['24092002', '24092013', '24092016', '24092020', '24092024', '24092026', '24092030']
Predict/Test: ['24092009', '24092014', '24092017', '24092023', '24092025', '24092029', '24092031']
  exit=0, duration=3274.06s, mean_dice=None

FINAL RESULTS - ribosome 2-fold
                fold_name  exit_code  duration_sec mean_dice
0  fold1_train_fold2_test          0       3203.26      None
1  fold2_train_fold1_test          0       3274.06      None

Tot